# 05 — Model Comparison: DDPM vs Rectified Flow

Side-by-side evaluation of all trained generative models on the same real test set.

**Requires** (run before this notebook):
- `02_clustering.ipynb` — produces `clusters.csv`
- `03_diffusion_training.ipynb` (full GPU run) — produces `best_model.pkl`
- `03b_rectified_flow_training.ipynb` (full GPU run) — produces `rf_best_model.pkl`

**Outputs**:
- `data/comparison_metrics.csv` — summary table (models × conditions × metrics)
- `data/comparison_figures/` — per-condition plots saved as PNG

**Metrics compared**: Discriminative accuracy · CRPS · ACF L2 · Marginal Wasserstein

In [ ]:
# ── Environment bootstrap ────────────────────────────────────────────────────
import shutil, subprocess, sys, tempfile, urllib.request, zipfile
from pathlib import Path


def find_or_bootstrap_repo_root():
    candidates = [
        Path.cwd().resolve(),
        Path('/tmp/vscode-colab/tesina'),
        Path('/content/tesina'),
        Path('/home/nicola/Desktop/Supsi/tesina'),
    ]
    for base in candidates:
        for candidate in [base, *base.parents]:
            if (candidate / 'src').exists() and (candidate / 'data').exists():
                return candidate

    runtime_dir  = Path(tempfile.gettempdir()) / 'vscode-colab'
    repo_dir     = runtime_dir / 'tesina'
    archive_path = runtime_dir / 'tesina.zip'
    extract_dir  = runtime_dir / 'tesina-master'
    repo_url     = 'https://github.com/ncapac/tesina.git'
    archive_url  = 'https://codeload.github.com/ncapac/tesina/zip/refs/heads/master'
    runtime_dir.mkdir(parents=True, exist_ok=True)
    if repo_dir.exists() and not (repo_dir / 'src').exists():
        shutil.rmtree(repo_dir)
    if not repo_dir.exists():
        try:
            result = subprocess.run(
                ['git', 'clone', '--depth', '1', repo_url, str(repo_dir)],
                capture_output=True, text=True,
            )
            if result.returncode != 0:
                raise RuntimeError(result.stderr)
        except Exception:
            if archive_path.exists(): archive_path.unlink()
            if extract_dir.exists():  shutil.rmtree(extract_dir)
            urllib.request.urlretrieve(archive_url, archive_path)
            with zipfile.ZipFile(archive_path, 'r') as z: z.extractall(runtime_dir)
            if repo_dir.exists(): shutil.rmtree(repo_dir)
            shutil.move(str(extract_dir), str(repo_dir))
    if (repo_dir / 'src').exists() and (repo_dir / 'data').exists():
        return repo_dir
    raise RuntimeError('Could not locate or bootstrap the tesina project root.')


REPO_ROOT      = find_or_bootstrap_repo_root()
DATA_DIR       = REPO_ROOT / 'data'
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints'
FIGURES_DIR    = DATA_DIR / 'comparison_figures'
FIGURES_DIR.mkdir(exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import equinox as eqx

from src.data.loader           import load_raw, compute_stats, normalize, denormalize
from src.data.dataset          import make_windows, train_val_split, numpy_dataloader
from src.models.transformer1d  import DiffusionTransformer1D
from src.models.diffusion      import DiffusionProcess
from src.models.rectified_flow import RectifiedFlowProcess
from src.evaluation.metrics    import (
    compare_models, run_all_metrics, marginal_wasserstein,
    acf_compare, crps_score, discriminative_score,
)

plt.rcParams['figure.dpi'] = 110
print('Project root:', REPO_ROOT)
print('JAX devices :', jax.devices())

## 1. Load data and checkpoints

In [ ]:
# ── Data ─────────────────────────────────────────────────────────────────────
df = load_raw(DATA_DIR / 'power.pk')
clusters_df    = pd.read_csv(DATA_DIR / 'clusters.csv')
cluster_labels = clusters_df['cluster_id'].values
N_CLUSTERS     = int(cluster_labels.max()) + 1

timestamps = df.index if isinstance(df.index, pd.DatetimeIndex) else None
stats      = compute_stats(df, cluster_labels)
df_norm    = normalize(df, stats, cluster_labels)

xs, cs, mid = make_windows(df_norm, cluster_labels, timestamps)
_, _, x_val, c_val = train_val_split(xs, cs, mid, n_meters=df.shape[1])
print(f'Val set: {x_val.shape[0]} windows')

print('\nVal windows per condition:')
for cid in range(N_CLUSTERS):
    for dt, dn in [(0, 'weekday'), (1, 'weekend')]:
        n = ((c_val[:, 0] == cid) & (c_val[:, 1] == dt)).sum()
        print(f'  cluster{cid}_{dn}: {n}')

In [ ]:
# ── Load checkpoints ─────────────────────────────────────────────────────────
import pickle

DDPM_CKPT = CHECKPOINT_DIR / 'best_model.pkl'
RF_CKPT   = CHECKPOINT_DIR / 'rf_best_model.pkl'

for p in [DDPM_CKPT, RF_CKPT]:
    if not p.exists():
        print(f'⚠  Missing checkpoint: {p}')
        print('   Run the corresponding training notebook on GPU first.')

with open(DDPM_CKPT, 'rb') as f:
    ddpm_ckpt = pickle.load(f)
ddpm_model = ddpm_ckpt['model']
print(f'DDPM checkpoint loaded  (step {ddpm_ckpt["step"]})')

with open(RF_CKPT, 'rb') as f:
    rf_ckpt = pickle.load(f)
rf_model = rf_ckpt['model']
print(f'RF   checkpoint loaded  (step {rf_ckpt["step"]})')

# ── Instantiate processes ─────────────────────────────────────────────────────
diffusion = DiffusionProcess(T=1000, freq_loss_weight=0.05)
rf_proc   = RectifiedFlowProcess(freq_loss_weight=0.05)

## 2. Build generator functions

Each generator accepts `(c_batch: np.ndarray, seed: int) -> np.ndarray (N, L)`.
The `compare_models()` framework calls these for every condition.

In [ ]:
N_SAMPLES      = 200     # synthetic samples per condition per model
GUIDANCE_SCALE = 1.5
N_DDIM_STEPS   = 50


def make_ddpm_generator(model, diff):
    def generate(c_batch, seed):
        key = jax.random.PRNGKey(int(seed))
        c   = jnp.array(c_batch, dtype=jnp.int32)
        return diff.ddim_sample(
            model, c,
            seq_len=24, batch_size=len(c_batch),
            key=key, n_steps=N_DDIM_STEPS, guidance_scale=GUIDANCE_SCALE,
        )
    return generate


def make_rf_generator(model, rf):
    def generate(c_batch, seed):
        key = jax.random.PRNGKey(int(seed))
        c   = jnp.array(c_batch, dtype=jnp.int32)
        return rf.sample(
            model, c,
            seq_len=24, batch_size=len(c_batch),
            key=key, n_steps=N_DDIM_STEPS, guidance_scale=GUIDANCE_SCALE,
        )
    return generate


models_dict = {
    'DDPM': make_ddpm_generator(ddpm_model, diffusion),
    'RF':   make_rf_generator(rf_model, rf_proc),
}

print(f'Generator functions ready: {list(models_dict.keys())}')
print(f'Samples per condition: {N_SAMPLES}  |  Guidance scale: {GUIDANCE_SCALE}')

## 3. Run comparison

In [ ]:
summary_df, figs_dict = compare_models(
    models_dict        = models_dict,
    real_data          = x_val,
    conditions         = c_val,
    n_samples          = N_SAMPLES,
    guidance_scale     = GUIDANCE_SCALE,
    n_ddim_steps       = N_DDIM_STEPS,
    seed               = 0,
    show_figs          = False,  # set True to display per-condition figures inline
    verbose            = True,
)

print('\n── Summary ──')
print(summary_df.to_string(index=False))

## 4. Summary metric table

In [ ]:
metric_cols = ['discriminative_acc', 'crps', 'acf_l2', 'wasserstein']

# Pivot: rows = condition, cols = (model, metric)
pivot = summary_df.pivot_table(
    index='condition', columns='model', values=metric_cols,
)
print('Comparison table:')
print(pivot.round(4).to_string())

# Save CSV
csv_path = DATA_DIR / 'comparison_metrics.csv'
summary_df.to_csv(csv_path, index=False)
print(f'\nSaved -> {csv_path}')

In [ ]:
# ── Bar chart: discriminative accuracy per condition per model ────────────────
conditions_ordered = sorted(summary_df['condition'].unique())
n_cond = len(conditions_ordered)
x_pos  = np.arange(n_cond)
width  = 0.35
colors_m = {'DDPM': 'steelblue', 'RF': 'coral'}

fig, ax = plt.subplots(figsize=(max(10, n_cond * 1.6), 4.5))
for i, model_name in enumerate(['DDPM', 'RF']):
    sub = summary_df[summary_df['model'] == model_name].set_index('condition')
    vals = [sub.loc[c, 'discriminative_acc'] if c in sub.index else float('nan')
            for c in conditions_ordered]
    ax.bar(x_pos + i * width - width/2, vals,
           width=width, label=model_name, color=colors_m[model_name], alpha=0.85, zorder=3)

# Quality bands
ax.axhspan(0.48, 0.52, color='green',  alpha=0.10, label='Excellent (<0.52)')
ax.axhspan(0.52, 0.60, color='yellow', alpha=0.15, label='Acceptable (<0.60)')
ax.axhspan(0.60, 1.00, color='red',    alpha=0.06, label='Poor (>0.60)')
ax.axhline(0.5,  color='black', ls='--', lw=1.2, label='Ideal (0.5)')

ax.set_xticks(x_pos); ax.set_xticklabels(conditions_ordered, rotation=30, ha='right', fontsize=8)
ax.set_ylim(0.4, 1.0); ax.set_ylabel('Discriminative accuracy')
ax.set_title('Discriminative score: DDPM vs RF  (target ≈ 0.5)')
ax.legend(ncol=3, fontsize=8, loc='upper right')
ax.grid(axis='y', alpha=0.3, zorder=0)
plt.tight_layout(); plt.show()
fig.savefig(FIGURES_DIR / 'discriminative_comparison.png', dpi=150, bbox_inches='tight')

In [ ]:
# ── Radar / grouped bar: all 4 metrics (normalised) ──────────────────────────
# Normalise each metric to [0,1] range across all values for easy visual comparison.
# For disc_acc, distance from 0.5 is the meaningful quantity.

fig, axes = plt.subplots(1, len(metric_cols), figsize=(5 * len(metric_cols), 4.5), sharey=False)
colors_m2 = ['steelblue', 'coral']

for ax, metric in zip(axes, metric_cols):
    for i, model_name in enumerate(['DDPM', 'RF']):
        sub = summary_df[summary_df['model'] == model_name].set_index('condition')
        vals = [float(sub.loc[c, metric]) if c in sub.index else float('nan')
                for c in conditions_ordered]
        ax.bar(x_pos + i * width - width/2, vals,
               width=width, label=model_name, color=colors_m2[i], alpha=0.85, zorder=3)
    if metric == 'discriminative_acc':
        ax.axhline(0.5, color='black', ls='--', lw=1.0)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(conditions_ordered, rotation=40, ha='right', fontsize=7)
    ax.set_title(metric.replace('_', ' ').title(), fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle('All metrics: DDPM vs RF per condition', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
fig.savefig(FIGURES_DIR / 'all_metrics_comparison.png', dpi=150, bbox_inches='tight')

## 5. Per-condition visual comparison

In [ ]:
# ── Generate sample galleries for each condition ──────────────────────────────
# For each (cluster, day_type): side-by-side mean±σ profiles DDPM vs RF vs Real

hours = np.arange(24)

for cid in range(N_CLUSTERS):
    for dt, day_name in [(0, 'Weekday'), (1, 'Weekend')]:
        cond_label = f"cluster{cid}_{'weekday' if dt == 0 else 'weekend'}"
        rep_dow    = 1 if dt == 0 else 5

        # Real val windows for this condition
        mask_r    = (c_val[:, 0] == cid) & (c_val[:, 1] == dt)
        real_cond = x_val[mask_r]
        if len(real_cond) < 5:
            print(f'Skipping {cond_label}: only {len(real_cond)} real samples')
            continue

        c_batch = jnp.array([[cid, dt, 5, rep_dow]] * N_SAMPLES, dtype=jnp.int32)

        synth_ddpm = np.array(diffusion.ddim_sample(
            ddpm_model, c_batch, seq_len=24, batch_size=N_SAMPLES,
            key=jax.random.PRNGKey(cid * 10 + dt), n_steps=50, guidance_scale=GUIDANCE_SCALE,
        ))
        synth_rf = np.array(rf_proc.sample(
            rf_model, c_batch, seq_len=24, batch_size=N_SAMPLES,
            key=jax.random.PRNGKey(cid * 10 + dt + 50), n_steps=50, guidance_scale=GUIDANCE_SCALE,
        ))

        fig, ax = plt.subplots(figsize=(9, 3.5))
        for arr, name, col in [
            (real_cond, 'Real',  'steelblue'),
            (synth_ddpm, 'DDPM', 'coral'),
            (synth_rf,   'RF',   'mediumseagreen'),
        ]:
            mu, s = arr.mean(0), arr.std(0)
            ax.fill_between(hours, mu - s, mu + s, alpha=0.18, color=col)
            ax.plot(hours, mu, color=col, lw=2, label=f'{name} (n={len(arr)})')

        ax.set_xlabel('Hour of day'); ax.set_ylabel('Normalised consumption')
        ax.set_title(f'Mean ± σ  |  {cond_label}  |  DDPM vs RF vs Real')
        ax.legend(fontsize=8); ax.grid(alpha=0.25)
        plt.tight_layout()
        plt.show()
        fig.savefig(FIGURES_DIR / f'profile_{cond_label}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)

## 6. Training convergence comparison

In [ ]:
# ── Overlay DDPM and RF training loss curves ──────────────────────────────────
ddpm_tr = ddpm_ckpt.get('train_losses', [])
ddpm_va = ddpm_ckpt.get('val_losses',   [])
rf_tr   = rf_ckpt.get('train_losses',   [])
rf_va   = rf_ckpt.get('val_losses',     [])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for losses, name, col, ls in [
    (ddpm_tr, 'DDPM train', 'steelblue', '-'),
    (ddpm_va, 'DDPM val',   'steelblue', '--'),
    (rf_tr,   'RF train',   'coral',     '-'),
    (rf_va,   'RF val',     'coral',     '--'),
]:
    if losses:
        axes[0].plot(range(1, len(losses)+1), losses,
                     color=col, ls=ls, lw=1.5, label=name)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training loss: DDPM vs RF')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Relative improvement from epoch 1
for losses, name, col, ls in [
    (ddpm_tr, 'DDPM', 'steelblue', '-'),
    (rf_tr,   'RF',   'coral',     '-'),
]:
    if len(losses) > 1:
        rel = [(losses[0] - l) / losses[0] * 100 for l in losses]
        axes[1].plot(range(1, len(rel)+1), rel, color=col, ls=ls, lw=1.5, label=name)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('% improvement vs epoch 1')
axes[1].set_title('Convergence speed comparison')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
axes[1].axhline(0, color='grey', ls='--', lw=0.8)

plt.tight_layout(); plt.show()
fig.savefig(FIGURES_DIR / 'convergence_comparison.png', dpi=150, bbox_inches='tight')

## 7. Ablation: CFG guidance scale

In [ ]:
# ── Guidance scale sweep on Cluster 0 Weekday ────────────────────────────────
ABLATION_SCALES = [0.0, 0.5, 1.0, 1.5, 2.5, 4.0]
ABLATION_CID    = 0
ABLATION_DT     = 0     # weekday
ABLATION_N      = 150

mask_abl  = (c_val[:, 0] == ABLATION_CID) & (c_val[:, 1] == ABLATION_DT)
real_abl  = x_val[mask_abl]
c_abl_np  = np.array([[ABLATION_CID, ABLATION_DT, 5, 1]] * ABLATION_N, dtype=np.int32)

rows_abl = []
for model_name, generate_fn in models_dict.items():
    for scale in ABLATION_SCALES:
        # Temporarily build a generator with this specific scale
        c_jax = jnp.array(c_abl_np)
        key   = jax.random.PRNGKey(int(scale * 100))
        if model_name == 'DDPM':
            synth = np.array(diffusion.ddim_sample(
                ddpm_model, c_jax, seq_len=24, batch_size=ABLATION_N,
                key=key, n_steps=50, guidance_scale=scale,
            ))
        else:
            synth = np.array(rf_proc.sample(
                rf_model, c_jax, seq_len=24, batch_size=ABLATION_N,
                key=key, n_steps=50, guidance_scale=scale,
            ))
        disc = discriminative_score(real_abl, synth)
        crps = crps_score(real_abl, synth)
        acf  = acf_compare(real_abl, synth)
        wass = marginal_wasserstein(real_abl, synth)
        rows_abl.append({'model': model_name, 'scale': scale,
                         'disc': disc, 'crps': crps, 'acf_l2': acf, 'wass': wass})

abl_df = pd.DataFrame(rows_abl)
print(abl_df.to_string(index=False))

# ── Plot disc accuracy vs guidance scale ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for model_name, col in [('DDPM', 'steelblue'), ('RF', 'coral')]:
    sub = abl_df[abl_df['model'] == model_name]
    ax.plot(sub['scale'], sub['disc'], 'o-', color=col, lw=2, label=model_name)
ax.axhline(0.5, color='black', ls='--', lw=1.2, label='Ideal (0.5)')
ax.axhspan(0.48, 0.52, color='green',  alpha=0.10)
ax.axhspan(0.52, 0.60, color='yellow', alpha=0.10)
ax.set_xlabel('Guidance scale s'); ax.set_ylabel('Discriminative accuracy')
ax.set_title(f'CFG guidance scale ablation  —  C{ABLATION_CID} Weekday')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
fig.savefig(FIGURES_DIR / 'guidance_scale_ablation.png', dpi=150, bbox_inches='tight')

## 8. Conditioning ablation

How much does each conditioning feature contribute to sample quality?
Compare: cluster-only  vs  cluster+day_type  vs  full (cluster+day_type+month+dow).

In [ ]:
# ── Conditioning ablation on Cluster 0 Weekday ───────────────────────────────
# Achieved by zeroing out features in c_batch (setting to -1 = null token)
# Full:         [cid, dt, month, dow]
# No dow/month: [cid, dt, -1, -1]
# No day_type:  [cid, -1, -1, -1]
# Null (uncond):[−1, -1, -1, -1]

ablation_configs = {
    'full':               (ABLATION_CID, ABLATION_DT, 5, 1),
    'no_dow_month':       (ABLATION_CID, ABLATION_DT, -1, -1),
    'cluster_only':       (ABLATION_CID, -1, -1, -1),
    'unconditional':      (-1, -1, -1, -1),
}

rows_cond = []
for model_name, generate_fn in models_dict.items():
    for config_name, c_vals in ablation_configs.items():
        c_jax = jnp.array([list(c_vals)] * ABLATION_N, dtype=jnp.int32)
        key   = jax.random.PRNGKey(42)
        if model_name == 'DDPM':
            synth = np.array(diffusion.ddim_sample(
                ddpm_model, c_jax, seq_len=24, batch_size=ABLATION_N,
                key=key, n_steps=50, guidance_scale=GUIDANCE_SCALE,
            ))
        else:
            synth = np.array(rf_proc.sample(
                rf_model, c_jax, seq_len=24, batch_size=ABLATION_N,
                key=key, n_steps=50, guidance_scale=GUIDANCE_SCALE,
            ))
        disc = discriminative_score(real_abl, synth)
        wass = marginal_wasserstein(real_abl, synth)
        rows_cond.append({'model': model_name, 'conditioning': config_name,
                          'disc': disc, 'wasserstein': wass})
        print(f'  {model_name:4s}  {config_name:<20s}  disc={disc:.3f}  wass={wass:.4f}')

cond_df = pd.DataFrame(rows_cond)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
config_labels = list(ablation_configs.keys())
xpos = np.arange(len(config_labels))
width = 0.35
for i, (model_name, col) in enumerate([('DDPM', 'steelblue'), ('RF', 'coral')]):
    sub = cond_df[cond_df['model'] == model_name].set_index('conditioning')
    for ax_i, metric in [(0, 'disc'), (1, 'wasserstein')]:
        vals = [float(sub.loc[c, metric]) if c in sub.index else float('nan')
                for c in config_labels]
        axes[ax_i].bar(xpos + i * width - width/2, vals,
                       width=width, label=model_name, color=col, alpha=0.85)

for ax_i, (title, ideal_line) in enumerate([
    ('Discriminative acc (ideal=0.5)', 0.5),
    ('Wasserstein distance (lower=better)', None),
]):
    axes[ax_i].set_xticks(xpos)
    axes[ax_i].set_xticklabels(config_labels, rotation=20, ha='right', fontsize=8)
    axes[ax_i].set_title(title, fontsize=9)
    if ideal_line is not None:
        axes[ax_i].axhline(ideal_line, color='black', ls='--', lw=1.0, label='Ideal')
    axes[ax_i].legend(fontsize=8); axes[ax_i].grid(axis='y', alpha=0.3)

plt.suptitle('Conditioning ablation — C0 Weekday', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()
fig.savefig(FIGURES_DIR / 'conditioning_ablation.png', dpi=150, bbox_inches='tight')

## 9. Final summary & thesis notes

In [ ]:
print("=" * 65)
print("COMPARISON SUMMARY")
print("=" * 65)

for model_name in ['DDPM', 'RF']:
    sub = summary_df[summary_df['model'] == model_name]
    if sub.empty:
        continue
    print(f"\n── {model_name} ──")
    for _, row in sub.iterrows():
        offset  = abs(row['discriminative_acc'] - 0.5)
        quality = "excellent" if offset < 0.02 else ("good" if offset < 0.10 else
                  ("fair" if offset < 0.20 else "poor"))
        print(
            f"  {row['condition']:<35s}  "
            f"disc={row['discriminative_acc']:.3f} [{quality}]  "
            f"crps={row['crps']:.4f}  "
            f"acf={row['acf_l2']:.4f}  "
            f"wass={row['wasserstein']:.4f}"
        )

print("\n─" * 65)
print("THESIS NOTE — Weekday/Weekend separation:")
print("  Day-type signal is only ~1.4% in this dataset (see 01_eda §4b).")
print("  Expect near-identical weekday vs weekend metrics for both models.")
print("  This is the expected behaviour — not a model failure.")
print()
print("THESIS NOTE — DDPM vs RF comparison framing:")
print("  RF trains on velocity matching (continuous t) vs DDPM on noise")
print("  prediction (discrete t). With L=24 (very short sequences) and a")
print("  shared Transformer backbone, differences are expected to be small.")
print("  Key comparison axes: convergence speed, sample diversity (σ),")
print("  and sharpness of daily peak (ACF + per-timestep σ).")
print()
print(f"All figures saved to: {FIGURES_DIR}")
print(f"Metrics CSV saved to: {DATA_DIR / 'comparison_metrics.csv'}")